In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

print(nav.head())
print(nav.shape)
print(nav.columns.tolist())

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
(46000, 3)
['amfi_code', 'date', 'nav']


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── CLEAN NAV HISTORY ──────────────────────────────────────
nav = pd.read_csv(RAW / "02_nav_history.csv")

# 1. Parse dates
nav['date'] = pd.to_datetime(nav['date'])

# 2. Sort
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# 3. Forward-fill missing NAV for weekends/holidays
nav = (nav.groupby('amfi_code')
          .apply(lambda df: df.set_index('date')
                              .reindex(pd.date_range(df['date'].min(),
                                                     df['date'].max(), freq='B'))
                              .ffill())
          .drop(columns='amfi_code')
          .reset_index(names=['amfi_code','date']))

# 4. Remove duplicates
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])

# 5. Validate NAV > 0
invalid_nav = nav[nav['nav'] <= 0]
print(f"Invalid NAV rows (<=0): {len(invalid_nav)}")
nav = nav[nav['nav'] > 0]

print(f"✅ clean_nav shape: {nav.shape}")
nav.to_csv(PROCESSED / "clean_nav.csv", index=True)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_3360\3991996833.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.set_index('date')


Invalid NAV rows (<=0): 0
✅ clean_nav shape: (46000, 3)


In [5]:
from pathlib import Path
print(Path.cwd())

C:\Users\lenovo\bluestock_mf_capstone\notebooks


In [3]:
from pathlib import Path

print(PROCESSED.resolve())

C:\Users\lenovo\bluestock_mf_capstone\data\processed


In [4]:
import re

tx = pd.read_csv(RAW / "08_investor_transactions.csv")

# 1. Standardise transaction_type
tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()
valid_types = ['Sip', 'Lumpsum', 'Redemption']
print("Unique types before:", tx['transaction_type'].unique())
tx = tx[tx['transaction_type'].isin(valid_types)]

# 2. Validate amount > 0
tx = tx[tx['amount_inr'] > 0]

# 3. Fix date format
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])

# 4. Check KYC status
print("KYC values:", tx['kyc_status'].unique())
tx = tx[tx['kyc_status'].isin(['Verified', 'Pending'])]

print(f"✅ clean_transactions shape: {tx.shape}")
tx.to_csv(PROCESSED / "clean_transactions.csv", index=False)

Unique types before: ['Sip' 'Redemption' 'Lumpsum']
KYC values: ['Verified' 'Pending']
✅ clean_transactions shape: (32778, 13)


In [5]:
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

print(perf.shape)
print(perf.columns.tolist())

(40, 19)
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [6]:
perf = pd.read_csv(RAW / "07_scheme_performance.csv")

# 1. Validate return columns are numeric
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
               'sharpe_ratio', 'alpha', 'beta', 'max_drawdown_pct']
for col in return_cols:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

# 2. Flag negative Sharpe ratios (anomaly — don't delete, just flag)
perf['sharpe_flag'] = perf['sharpe_ratio'] < 0
print(f"Negative Sharpe count: {perf['sharpe_flag'].sum()}")

# 3. Validate expense ratio range 0.1% to 2.5%
out_of_range = perf[~perf['expense_ratio_pct'].between(0.1, 2.5)]
print(f"Expense ratio out of range: {len(out_of_range)}")

print(f"✅ clean_performance shape: {perf.shape}")
perf.to_csv(PROCESSED / "clean_performance.csv", index=False)

Negative Sharpe count: 0
Expense ratio out of range: 0
✅ clean_performance shape: (40, 20)
